# Total Probability and Bayes Theorem

**Probability and Distribution Theory I** · [Subject index](00_Subject_Index.ipynb)

## Learning route

1. Read the theory and definitions in this notebook.
2. Explain the symbols and assumptions.
3. Predict, calculate, then run the simulation.
4. Change one parameter and explain the result.
5. Complete the practice before revealing answers.

![Visual route](../_images/probability.svg)

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Partitions](#theory-01)
- [Total probability](#theory-02)
- [Bayesian updating](#theory-03)
- [Base rates and diagnostic examples](#theory-04)
- [Complement evidence and repeated updates](#theory-05)
- [Validation and interpretation](#theory-06)

## Detailed theory · Total probability and Bayes' theorem

<a id="theory-01"></a>

### Partitions
Events $B_1,\ldots,B_k$ form a partition when they are disjoint and cover S. For conditional formulas, use positive-probability partition elements. Sources, production lines or population groups can form a partition if every case belongs to exactly one.

<a id="theory-02"></a>

### Total probability
For evidence E,
$$P(E)=\sum_iP(E\mid B_i)P(B_i).$$
Each term is a joint source-and-evidence path. The denominator is not the unweighted average of source-specific rates unless source probabilities are equal.

<a id="theory-03"></a>

### Bayesian updating
$$P(B_j\mid E)=\frac{P(E\mid B_j)P(B_j)}{\sum_iP(E\mid B_i)P(B_i)}.$$
The prior is the source probability before E; the likelihood is the evidence probability within a source; the posterior is the normalized probability after E. This elementary identity is valid under any consistent joint probability model.

<a id="theory-04"></a>

### Base rates and diagnostic examples
A rare source can remain unlikely even with a high evidence rate. Sensitivity $P(+\mid condition)$ is not $P(condition\mid+)$. The latter also needs prevalence and the false-positive rate. Always define what each conditional event means.

<a id="theory-05"></a>

### Complement evidence and repeated updates
For no repair or a negative result, use the corresponding source-specific complement rates and recompute the total evidence probability. Successive updates need the likelihood conditional on previous information; multiplying marginal likelihoods assumes conditional independence.

<a id="theory-06"></a>

### Validation and interpretation
The posterior probabilities across all sources sum to one. Very small denominators can make estimates sensitive to uncertain input rates. A posterior is conditional on the stated model and data; it does not automatically justify a causal claim or an individual certainty.

## Concepts covered

**Source alignment:** Batch 5: Total probability and Bayes’ theorem (source pp. 10–11).

**Mission:** Learn the rule, predict the result, and use code to explore it.

## Worked calculation · Bayes theorem: prior-to-posterior table

### Question and assumptions

Factory A produces 60% of items with defect probability .02; B produces 40% with defect probability .05. Given a defective item, find the probability it came from B.

### Procedure

1. Multiply each prior by its conditional defect rate.
2. Add the joint defect probabilities to obtain total defect probability.
3. Normalize each joint probability by that total.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Factory | Prior | P(defect | factory) | Joint defect probability | Posterior given defect |
| --- | --- | --- | --- | --- |
| A | 0.6 | 0.02 | 0.012 | 0.375 |
| B | 0.4 | 0.05 | 0.02 | 0.625 |

| Quantity | Value |
| --- | --- |
| P(defect) | 0.032 |
| P(B / defect) | 0.625 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


prior=[.6,.4]; rate=[.02,.05]; joint=[p*r for p,r in zip(prior,rate)]; total=sum(joint)
rows=[[g,p,r,j,j/total] for g,p,r,j in zip(["A","B"],prior,rate,joint)]
show_table(["Factory","Prior","P(defect | factory)","Joint defect probability","Posterior given defect"],rows)
show_table(["Quantity","Value"],[["P(defect)",total],["P(B | defect)",joint[1]/total]])
assert abs(sum(row[-1] for row in rows)-1)<1e-12


### Decision and interpretation

B has a lower production share but a higher posterior share among defective items. The posterior denominator must include defects from both factories.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from itertools import product
rng=np.random.default_rng(42)

In [ ]:
prior=np.array([.50,.30,.20]); repair=np.array([.25,.20,.10])
paths=prior*repair; p_repair=paths.sum(); posterior=paths/p_repair
print('repair path probabilities:',paths)
print('total P(repair):',p_repair)
print('P(brand | repair):',posterior,'sum:',posterior.sum())
# natural frequencies per 1000 devices
print('per 1000: repaired by source:',1000*paths,'total repairs:',1000*p_repair)

### Practice · Bayes detective
1. Brands A/B/C have shares (.50,.30,.20) and repair rates (.25,.20,.10). Find total repair probability and P(A | repair).
2. Find P(B | repair), P(C | repair), and verify the posterior probabilities sum to one.
3. A disease has prevalence .02; sensitivity .90; specificity .95. Find P(disease | positive).

<details><summary>Answer sketches</summary>
1. Total=.50(.25)+.30(.20)+.20(.10)=.205. P(A|repair)=.125/.205≈.6098. 2. .060/.205≈.2927; .020/.205≈.0976; sum≈1 after rounding. 3. .90(.02)/[.90(.02)+.05(.98)]≈.269.
</details>